In [1]:
import os
import numpy as np
import csv
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
import math
import time
import nltk
import tensorflow as tf
from tensorflow.keras.layers import GRU, LSTM, Bidirectional, Dense, Flatten, Conv1D, BatchNormalization, LeakyReLU, Dropout
from tensorflow.keras import Sequential
from tensorflow.keras.utils import plot_model
from pickle import load
from sklearn.metrics import mean_squared_error
from tqdm import tqdm
import statsmodels.api as sm
from math import sqrt
from datetime import datetime, timedelta
from sklearn.preprocessing import MinMaxScaler
from pickle import dump
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import unicodedata
import plotly.express as px
import warnings
warnings.filterwarnings("ignore")

In [2]:
all_tweets = pd.read_csv('stock_tweets.csv')

In [3]:
print(all_tweets.shape)
all_tweets['Stock Name'].unique()

(80793, 4)


array(['TSLA', 'MSFT', 'PG', 'META', 'AMZN', 'GOOG', 'AMD', 'AAPL',
       'NFLX', 'TSM', 'KO', 'F', 'COST', 'DIS', 'VZ', 'CRM', 'INTC', 'BA',
       'BX', 'NOC', 'PYPL', 'ENPH', 'NIO', 'ZS', 'XPEV'], dtype=object)

In [4]:
df = all_tweets #[all_tweets['Stock Name'].isin(stock_names)]
print(df.shape)
df.head()

(80793, 4)


,Date,Tweet,Stock Name,Company Name
0,2022-09-29 23:41:16+00:00,Mainstream media has done an amazing job at br...,TSLA,"Tesla, Inc."
1,2022-09-29 23:24:43+00:00,Tesla delivery estimates are at around 364k fr...,TSLA,"Tesla, Inc."
2,2022-09-29 23:18:08+00:00,3/ Even if I include 63.0M unvested RSUs as of...,TSLA,"Tesla, Inc."
3,2022-09-29 22:40:07+00:00,@RealDanODowd @WholeMarsBlog @Tesla Hahaha why...,TSLA,"Tesla, Inc."
4,2022-09-29 22:27:05+00:00,"@RealDanODowd @Tesla Stop trying to kill kids,...",TSLA,"Tesla, Inc."


In [5]:
df.info()
df.describe()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80793 entries, 0 to 80792
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Date          80793 non-null  object
 1   Tweet         80793 non-null  object
 2   Stock Name    80793 non-null  object
 3   Company Name  80793 non-null  object
dtypes: object(4)
memory usage: 2.5+ MB


Date            0
Tweet           0
Stock Name      0
Company Name    0
dtype: int64

In [6]:
df['Stock Name'].value_counts()

Stock Name
TSLA    37422
TSM     11034
AAPL     5056
PG       4089
AMZN     4089
MSFT     4089
NIO      3021
META     2751
AMD      2227
NFLX     1727
GOOG     1291
PYPL      843
DIS       635
BA        399
COST      393
INTC      315
KO        310
CRM       233
XPEV      225
ENPH      216
ZS        193
VZ        123
BX         50
NOC        31
F          31
Name: count, dtype: int64

In [7]:
sent_df = df.copy()
sent_df["sentiment_score"] = ''
sent_df["Negative"] = ''
sent_df["Neutral"] = ''
sent_df["Positive"] = ''
sent_df.head()

,Date,Tweet,Stock Name,Company Name,sentiment_score,Negative,Neutral,Positive
0,2022-09-29 23:41:16+00:00,Mainstream media has done an amazing job at br...,TSLA,"Tesla, Inc.",,,,
1,2022-09-29 23:24:43+00:00,Tesla delivery estimates are at around 364k fr...,TSLA,"Tesla, Inc.",,,,
2,2022-09-29 23:18:08+00:00,3/ Even if I include 63.0M unvested RSUs as of...,TSLA,"Tesla, Inc.",,,,
3,2022-09-29 22:40:07+00:00,@RealDanODowd @WholeMarsBlog @Tesla Hahaha why...,TSLA,"Tesla, Inc.",,,,
4,2022-09-29 22:27:05+00:00,"@RealDanODowd @Tesla Stop trying to kill kids,...",TSLA,"Tesla, Inc.",,,,


To get sentiment (polarity) scores, we use **VADER (Valence Aware Dictionary for Sentiment Reasoning)** model. VADER is a model used for text sentiment analysis that is sensitive to both polarity (positive/negative) and intensity (strength) of emotion. It is available in the NLTK package and can be applied directly to unlabeled text data.

VADER sentimental analysis relies on a dictionary that maps lexical features to emotion intensities known as sentiment scores. The sentiment score of a text can be obtained by summing up the intensity of each word in the text.

In [8]:
%%time
sentiment_analyzer = SentimentIntensityAnalyzer()
for indx, row in sent_df.iterrows():
    try:
        sentence_i = unicodedata.normalize('NFKD', sent_df.loc[indx, 'Tweet'])
        sentence_sentiment = sentiment_analyzer.polarity_scores(sentence_i)
        sent_df.at[indx, 'sentiment_score'] = sentence_sentiment['compound']
        sent_df.at[indx, 'Negative'] = sentence_sentiment['neg']
        sent_df.at[indx, 'Neutral'] = sentence_sentiment['neu']
        sent_df.at[indx, 'Positive'] = sentence_sentiment['pos']
    except TypeError:
        print (sent_df.loc[indx, 'Tweet'])
        print (indx)
        break

CPU times: user 13.4 s, sys: 164 ms, total: 13.5 s
Wall time: 13.8 s


In [9]:
sent_df = sent_df.drop(columns=['Company Name'])

In [10]:
sent_df.head()

,Date,Tweet,Stock Name,sentiment_score,Negative,Neutral,Positive
0,2022-09-29 23:41:16+00:00,Mainstream media has done an amazing job at br...,TSLA,0.0772,0.127,0.758,0.115
1,2022-09-29 23:24:43+00:00,Tesla delivery estimates are at around 364k fr...,TSLA,0.0,0.0,1.0,0.0
2,2022-09-29 23:18:08+00:00,3/ Even if I include 63.0M unvested RSUs as of...,TSLA,0.296,0.0,0.951,0.049
3,2022-09-29 22:40:07+00:00,@RealDanODowd @WholeMarsBlog @Tesla Hahaha why...,TSLA,-0.7568,0.273,0.59,0.137
4,2022-09-29 22:27:05+00:00,"@RealDanODowd @Tesla Stop trying to kill kids,...",TSLA,-0.875,0.526,0.474,0.0


In [11]:
# Define a function to filter DataFrame based on time range
def filter_time_range(df, start_time, end_time):
    df['Date'] = pd.to_datetime(df['Date'])
    df['Time'] = df['Date'].dt.time
    df['DateTime'] = df['Date'].dt.strftime('%Y-%m-%d') + ' ' + df['Time'].astype(str)
    df['DateTime'] = pd.to_datetime(df['DateTime'])
    return df[(df['Time'] >= start_time) & (df['Time'] <= end_time)]

# Define the time ranges in EST
start_time_est = pd.Timestamp('05:30:00').time()
end_time_est = pd.Timestamp('12:30:00').time()

# Filter DataFrame for two different time ranges
df_within_time_range = filter_time_range(sent_df, start_time_est, end_time_est)
df_outside_time_range = sent_df[~sent_df.index.isin(df_within_time_range.index)]

df_within_time_range['Date'] = pd.to_datetime(df_within_time_range['Date'])
df_within_time_range['Date'] = df_within_time_range['Date'].dt.date

df_outside_time_range['Date'] = pd.to_datetime(df_outside_time_range['Date'])
df_outside_time_range['Date'] = df_outside_time_range['Date'].dt.date

# Aggregate sentiment scores for both time ranges
result_within_time_range = df_within_time_range.groupby(['Stock Name', 'Date']).agg({'sentiment_score': ['mean', 'min', 'max', lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)]}).reset_index()
result_within_time_range.columns = ['Stock Name', 'Date', 'Mean', 'Min', 'Max', '25th Percentile', '75th Percentile']

result_outside_time_range = df_outside_time_range.groupby(['Stock Name', 'Date']).agg({'sentiment_score': ['mean', 'min', 'max', lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)]}).reset_index()
result_outside_time_range.columns = ['Stock Name', 'Date', 'Mean', 'Min', 'Max', '25th Percentile', '75th Percentile']

print("Results for time range between 5:30 AM EST to 12:30 PM EST:")
print(result_within_time_range)

print("\nResults for time range outside 5:30 AM EST to 12:30 PM EST:")
print(result_outside_time_range)


Results for time range between 5:30 AM EST to 12:30 PM EST:
     Stock Name        Date    Mean     Min     Max  25th Percentile  \
0          AAPL  2021-09-30  0.2739  0.2296  0.3182          0.25175   
1          AAPL  2021-10-07  0.1484     0.0   0.368          0.03860   
2          AAPL  2021-10-08  0.7842  0.7778  0.7906          0.78100   
3          AAPL  2021-10-10  0.3612  0.3612  0.3612          0.36120   
4          AAPL  2021-10-13 -0.4278 -0.4278 -0.4278         -0.42780   
...         ...         ...     ...     ...     ...              ...   
2503         ZS  2022-08-30     0.0     0.0     0.0          0.00000   
2504         ZS  2022-09-01  0.7351  0.7351  0.7351          0.73510   
2505         ZS  2022-09-17  0.8271  0.8271  0.8271          0.82710   
2506         ZS  2022-09-21  0.5122  0.5122  0.5122          0.51220   
2507         ZS  2022-09-29     0.0     0.0     0.0          0.00000   

      75th Percentile  
0             0.29605  
1             0.22260  
2  